# Masked-Diffusion BabyLM — Strict-Small (Upload to HuggingFace)

Push a finished training run to the Hub. The final checkpoint goes to `main`;
every required intermediate checkpoint becomes a `chck_NM` branch
(`chck_1M ... chck_10M, chck_20M ... chck_100M`). The custom modeling code and
tokenizer are bundled so `trust_remote_code=True` works for the evaluators.

**Needs an `HF_TOKEN` with write access.**

In [ ]:
# Cell 1 — Parameters
REPO_URL   = "https://github.com/Amos-Luna/Masked-Diffusion-BabyLM.git"
DRIVE_ROOT = "/content/drive/MyDrive/Researchs/BabyLM_diffusion_G4"
CONDITION  = "MD_base"   # MD_base | MD_freq_mask | MD_layerdup
SEED       = 42
REPO_ID    = "amosluna/babylm-2026-strict-small-mdlm-seed42"
TOKENIZER  = f"{DRIVE_ROOT}/tokenizer/mdlm_bpe_16k"

# Which run (day) to upload. Leave None to auto-pick the latest run on Drive for
# (CONDITION, SEED); the next cell lists every available run so you can instead
# set this MANUALLY to one of the printed paths, e.g.
# RUN_DIR = f"{DRIVE_ROOT}/runs/2026-06-08_MD_base_seed42"
RUN_DIR    = None

In [ ]:
# Cell 2 — Mount Drive, clone repo, install HF SDK, set token
from google.colab import drive; drive.mount("/content/drive")
import os, subprocess
if not os.path.isdir("/content/Masked-Diffusion-BabyLM"):
    subprocess.run(["git", "clone", REPO_URL, "/content/Masked-Diffusion-BabyLM"], check=True)
%cd /content/Masked-Diffusion-BabyLM/diffusion
!pip install -q huggingface_hub
try:
    from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    print("Set HF_TOKEN (write) in Colab Secrets.", e)

In [ ]:
# Cell 3 — List available runs on Drive (days auto-detected) and resolve RUN_DIR
# Runs are saved by train.py as runs/{YYYY-MM-DD}_{condition}_seed{S}/ (the date
# is detected automatically). This cell prints every run that has checkpoints so
# you can pick one MANUALLY: set RUN_DIR in Cell 1 to a printed path. If you
# leave RUN_DIR=None, it auto-selects the latest run matching (CONDITION, SEED).
import glob, json, os

runs_root = f"{DRIVE_ROOT}/runs"
all_runs = sorted(d for d in glob.glob(f"{runs_root}/*") if os.path.isdir(os.path.join(d, "checkpoints")))
print(f"Available runs in {runs_root}:\n")
for d in all_runs:
    ckpts = sorted(glob.glob(f"{d}/checkpoints/step_*"))
    words = "?"
    try:
        words = json.load(open(f"{d}/summary.json")).get("words_seen_total", "?")
    except Exception:
        pass
    print(f"  {os.path.basename(d):40s}  {len(ckpts):>3d} ckpts  words_seen={words}")

if RUN_DIR is None:
    matches = sorted(d for d in all_runs if d.endswith(f"_{CONDITION}_seed{SEED}"))
    if not matches:
        raise SystemExit(f"\nNo run found for ({CONDITION}, seed{SEED}). Set RUN_DIR manually above.")
    RUN_DIR = matches[-1]
    print(f"\nRUN_DIR=None -> auto-picked latest for ({CONDITION}, seed{SEED}):")
print(f"\n>> Uploading run: {RUN_DIR}")
assert os.path.isdir(RUN_DIR), f"RUN_DIR does not exist: {RUN_DIR}"

In [ ]:
# Cell 4 — Dry-run first (lists what would be pushed; uploads nothing)
!python scripts/upload_to_hf.py --run-dir "$RUN_DIR" --repo-id $REPO_ID \
    --tokenizer-dir "$TOKENIZER" --condition $CONDITION --seed $SEED --dry-run

In [ ]:
# Cell 5 — Real upload (final -> main, intermediates -> chck_NM branches)
!python scripts/upload_to_hf.py --run-dir "$RUN_DIR" --repo-id $REPO_ID \
    --tokenizer-dir "$TOKENIZER" --condition $CONDITION --seed $SEED